# EDA — APERTURE Finanzas Personales

Inspección de originales sin entrenamiento ni modificación de fuentes. Ejecutar con el entorno del proyecto. Las conclusiones verificadas están en `docs/eda_inicial.md`.


In [ ]:
from pathlib import Path
import sys
raiz = Path.cwd()
if not (raiz / "scripts").exists():
    raiz = raiz.parent
sys.path.insert(0, str(raiz))
from scripts.eda_datasets import audit
import pandas as pd
import plotly.express as px
archivos = [next((raiz/"data/demo").rglob("expense_data_1.csv")), next((raiz/"data/demo").rglob("personal_transactions*.xlsx"))]
resultados = [audit(p) for p in archivos]


## Faltantes y tipos
Los campos vacíos opcionales se distinguen de fechas, importes y categorías esenciales. No se borran registros automáticamente.


In [ ]:
for resumen, faltantes, mensual, semanal, categorias, matriz in resultados:
    print(resumen["file"])
    display(pd.Series(resumen["missing"], name="faltantes").to_frame())
    display(pd.Series(resumen["type_counts"], name="movimientos").to_frame())


## Cobertura temporal
Una semana dentro del rango de fechas no está certificada como completa. Ausencia de registro no significa consumo cero.


In [ ]:
for resumen, faltantes, mensual, semanal, categorias, matriz in resultados:
    print(resumen["file"], resumen["date_min"], resumen["date_max"])
    display(mensual)
    grafico = semanal.reset_index(names="semana")
    grafico["semana"] = grafico.semana.astype(str)
    display(px.line(grafico, x="semana", y="expense_minor", color="inside_date_bounds", title=resumen["file"] + " — unidades menores de análisis"))


## Categorías, actividad y atípicos
La frecuencia se estudia además del importe. Las banderas IQR no equivalen a errores ni justifican eliminar transacciones.


In [ ]:
for resumen, faltantes, mensual, semanal, categorias, matriz in resultados:
    print(resumen["file"], resumen["analysis_unit"])
    display(categorias)


## Revisión semántica y viabilidad

- Conciliar pagos de tarjeta: no duplicar consumo.
- Confirmar la moneda del Excel.
- Revisar notas ambiguas del CSV sin inferir préstamos automáticamente.
- Separar perfiles y épocas.
- No seleccionar variables ni categorías con el periodo de prueba.
- Excel: candidato exploratorio; CSV: historial corto.
- La utilidad predictiva y la calibración no están evaluadas: no se entrenó ningún modelo.
